In [0]:
# Instead of merging constantly, we are going to just over write everything since our data set isn't that large

import pyspark.sql.functions as F
from pyspark.sql.types import *


In [0]:
%run "../utilities/Delta Merge"


In [0]:
silver_data_source = spark.read.table('lakehouse.`03_silver`.earthquake_events')

In [0]:
silver_data_source.display()

In [0]:
# TODO
# 1. For a given hour how many events occured and its max and min magnitude - grouping events by hour

In [0]:
# Our function we are calling from utilities has a hash_id for primary key, so we need to add another one to this gold layer using activty_hour and 'net'
seismic_activity = (
    silver_data_source
    .withColumn("activity_hour", F.date_trunc("hour", F.col("time"))) # -> new column called activity_hour,where hour is the same for a group of events
    .groupBy("activity_hour","net") # -> group by hour and net(all the locations)
    .agg(
        F.count("hash_id").alias("total_event"),
        F.avg("mag").alias("avg_mag"),
        F.max("mag").alias("max_mag")
    )
     .withColumn('hash_id', F.sha2(F.concat_ws('_', F.col('activity_hour'), F.col('net')),256).substr(0,15)) # -> using substring to shorten the hash
)

seismic_activity.display()

In [0]:
table_name = 'lakehouse.`04_gold`.seismic_activity'

In [0]:
if check_delta_table(table_name):
    print(f"{table_name} table exists in catalog, updating it")
    delta_merge(seismic_activity, table_name, "hash_id")
else:
    print(f"{table_name} table does not exist in catalog, writing it for first time")
    seismic_activity.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed", "true").saveAsTable(table_name)